In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv("/kaggle/input/titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/titanic/test.csv")

In [ ]:
train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())
train_df['Embarked'] = train_df['Embarked'].fillna('S')

test_df['Age'] = test_df['Age'].fillna(test_df['Age'].median())
test_df['Fare'] = test_df['Fare'].fillna(test_df['Fare'].median())

In [ ]:
for df in [train_df, test_df]:
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'S':0, 'C':1, 'Q':2})

In [ ]:
for df in [train_df, test_df]:
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

In [ ]:
for df in [train_df, test_df]:
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')

for df in [train_df, test_df]:
    df['Title'] = df['Title'].map({"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3, "Rare": 4})

In [ ]:
for df in [train_df, test_df]:
    df['Title'] = df['Title'].map({"Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3, "Rare": 4})

In [ ]:
for df in [train_df, test_df]:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = 0
    df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1

In [ ]:
for df in [train_df, test_df]:
    df['CabinInitial'] = df['Cabin'].str[0]
    df['CabinInitial'] = df['CabinInitial'].fillna('U') 

In [ ]:
for df in [train_df, test_df]:
    df['CabinInitial'] = df['CabinInitial'].map({'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5,'F': 6, 'G': 7, 'T': 8, 'U': 0})

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
xgb = XGBClassifier(n_estimators=100, max_depth=4, random_state=42, use_label_encoder=False, eval_metric='logloss')

voting_model = VotingClassifier(estimators=[('rf', rf), ('gb', gb), ('xgb', xgb)], voting='soft')

In [ ]:
from sklearn.model_selection import cross_val_score

features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone', 'CabinInitial']

scores = cross_val_score(voting_model, train_df[features], train_df['Survived'], cv=5, scoring='accuracy')
print(scores)

In [ ]:
voting_model.fit(train_df[features], train_df['Survived'])
predictions = voting_model.predict(test_df[features])
output = pd.DataFrame({'PassengerId': test_df.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)

In [ ]:
print("success!")

In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# import pandas as pd

# # 🎯 Target variable
# y = train_data["Survived"]

# # 🧹 Handle missing values in both datasets
# for df in [train_data, test_data]:
#     df['Age'] = df['Age'].fillna(df['Age'].median())
#     df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
#     df['Fare'] = df['Fare'].fillna(df['Fare'].median())

#     # ✅ Create HasCabin safely
#     df['HasCabin'] = df['Cabin'].notnull().astype(int) if 'Cabin' in df.columns else 0

#     # ✅ Create FamilySize
#     df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# # 🧩 Updated feature list
# features = ["Pclass", "Sex", "Age", "Fare", "Embarked", "HasCabin", "FamilySize"]

# # 🔢 Convert categorical features to dummy variables
# X = pd.get_dummies(train_data[features])
# X_test = pd.get_dummies(test_data[features])

# # ⚖️ Align columns between train and test
# X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

# # 🌲 Train the model
# model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
# model.fit(X, y)

# # 🔮 Predict
# predictions = model.predict(X_test)

# # 💾 Save output
# output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
# output.to_csv('submission.csv', index=False)

# print("✅ Your submission was successfully saved!")


In [ ]:
# from sklearn.metrics import accuracy_score

# # Predict on training data
# train_preds = model.predict(X)

# # Compare predictions with actual labels
# train_acc = accuracy_score(y, train_preds)
# print("Training Accuracy:", train_acc)
